In [1]:
import numpy as  np
import pandas as pd

# Import functions for interpolation
from scipy.interpolate import LinearNDInterpolator 
from scipy.optimize import minimize

In [2]:
import pandas as pd

# Load the file, skipping the metadata header lines (usually starts with #)
# comment='#' tells pandas to ignore lines starting with that character
iso_df = pd.read_csv('output170031159724.dat.txt', sep='\s+', comment='#')
iso_df = iso_df.drop_duplicates(subset=['Mini', 'logAge', 'MH'])

# 1. Focus on the relevant Mass range (HD 209458 is ~1.15)
iso_df = iso_df[(iso_df['Mini'] >= 0.5) & (iso_df['Mini'] <= 2.5)]

# 2. Focus on relevant Age range (Stars aren't younger than 10Myr or older than 13Gyr)
iso_df = iso_df[(iso_df['logAge'] >= 7.0) & (iso_df['logAge'] <= 10.1)]

# 3. Keep only the Main Sequence and Subgiant phases (Label 1, 2, 3)
# This removes complex White Dwarf and Red Giant branches that slow down triangulation
iso_df = iso_df[iso_df['label'].isin([1, 2, 3])]

print(f"Rows remaining after filtering: {len(iso_df)}")


<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
C:\Users\cwray\AppData\Local\Temp\ipykernel_26536\2654797121.py:5: SyntaxWarning: invalid escape sequence '\s'
  iso_df = pd.read_csv('output170031159724.dat.txt', sep='\s+', comment='#')


Rows remaining after filtering: 19303


In [3]:
def create_iso_interp_engine(iso_df):
    """
    Creates an interpolator for G, BP, and RP mags based on 
    Mini (Mass), logAge, and MH (Metallicity).
    """
    # Define our 'X, Y, Z' coordinates
    # We use 'Mini' (Initial Mass) because it's the independent variable
    points = iso_df[['Mini', 'logAge', 'MH']].values
    
    # Create interpolators for each band
    print("Building interpolators... this may take a moment.")
    interp_G  = LinearNDInterpolator(points, iso_df['Gmag'].values)
    interp_BP = LinearNDInterpolator(points, iso_df['G_BPmag'].values)
    interp_RP = LinearNDInterpolator(points, iso_df['G_RPmag'].values)
    
    # Store them in a dictionary for easy access
    return {
        'G_mag': interp_G,
        'BP_mag': interp_BP,
        'RP_mag': interp_RP
    }

# Initialize the global engine
iso_engines = create_iso_interp_engine(iso_df)

Building interpolators... this may take a moment.


In [4]:
def iso_interp(params):
    """
    params: [mass, log_age, feh]
    Returns a dictionary of predicted absolute magnitudes.
    """
    mass, log_age, feh = params
    
    results = {}
    # Use the keys that match your observed_props dictionary
    for key in ['G_mag', 'BP_mag', 'RP_mag']:
        # Call the specific engine for this band
        val = iso_engines[key](mass, log_age, feh)
        
        # If the point is outside the PARSEC grid, val will be NaN
        if np.isnan(val):
            raise ValueError("Parameters outside grid bounds")
            
        results[key] = float(val)
        
    return results

In [5]:
# Set up to read data from both master phtometry and master target lists
def read_star_row_from_csv(
    host_name: str,
    sigma_mag: bool = True,
    sigma_parallax: float = 0.1,
):
    master_csv_path = "ASTR502_Mega_Target_List_1.csv"
    master_phot_csv_path = "ASTR502_Master_Photometry_List_1.csv"
    
    master_df = pd.read_csv(master_csv_path)
    phot_df = pd.read_csv(master_phot_csv_path)
    index = master_df[master_df["hostname"] == host_name].index[0]
    phot_index = phot_df[phot_df["hostname"] == host_name].index[0]
    # Select row
    row = master_df.iloc[index]
    phot_row = phot_df.iloc[phot_index]

    # Build props dict
    props = {}

    # --- Parallax (required) ---
    # Need this for magnitudes to mean anything. 
    if not np.isnan(row["st_parallax_mas"]):
        props["parallax"] = (row["st_parallax_mas"], sigma_parallax)

    # band map from master_phot_csv to MIST keys
    band_map = {
        "gaiaGmag": "G_mag",
        "giaaBPmag": "BP_mag",
        "gaiaRPmag": "RP_mag",
        "Jmag": "J_mag",
        "Hmag": "H_mag",
        "Kmag": "K_mag",
    }
    #get photometry from phot_df
    for csv_col, iso_key in band_map.items():
        if csv_col in phot_row and not np.isnan(phot_row[csv_col]):
            sigmamag = 0.02 if sigma_mag is False else phot_row[f"e_{csv_col}"]
            props[iso_key] = phot_row[csv_col]
            props[f"{iso_key}_err"] = sigmamag


    return props

In [6]:
# Load in the data using read_star_row_from_csv
sun = read_star_row_from_csv("HD 209458")
# Checking the data to make sure it was collected properly
print(sun)

{'parallax': (np.float64(20.7693898852824), 0.1), 'G_mag': np.float64(7.521244378), 'G_mag_err': np.float64(0.0183144879635296), 'RP_mag': np.float64(7.080287415), 'RP_mag_err': np.float64(0.0264984319384077), 'J_mag': np.float64(6.591), 'J_mag_err': np.float64(0.02), 'H_mag': np.float64(6.366), 'H_mag_err': np.float64(0.038), 'K_mag': np.float64(6.308), 'K_mag_err': np.float64(0.026)}


In [7]:
def objective_function(params, observed_props):
    mass, log_age, feh = params
    
    try:
        # This calls the function we just wrote above
        predicted_mags = iso_interp(params) 
    except (ValueError, Exception):
        # If the interpolator returns NaN or fails, return the penalty
        return 1e10 

    chi_sq = 0
    
    # Distance Modulus logic
    parallax_mas = observed_props['parallax'][0]
    dist_pc = 1000.0 / parallax_mas
    dist_mod = 5 * np.log10(dist_pc) - 5

    for band in ["G_mag", "BP_mag", "RP_mag"]:
        if band in observed_props:
            obs_mag = observed_props[band]
            obs_err = observed_props[f"{band}_err"]
            
            # PARSEC gives Absolute Mags, so we add Dist Mod to get Apparent
            pred_mag = predicted_mags[band] + dist_mod
            
            chi_sq += ((obs_mag - pred_mag) / obs_err)**2

    return chi_sq

In [8]:
def objective_function(params, observed_props):
    mass, log_age, feh = params
    try:
        predicted_mags = iso_interp([mass, log_age, feh]) 
    except:
        # This will tell you exactly which numbers are breaking the interpolator
        print(f"OUT OF BOUNDS: M={mass:.2f}, Age={log_age:.2f}, [Fe/H]={feh:.2f}")
        return 1e10

In [9]:
# Optimization for age estimation
def estimate_age(star_name, initial_guess, bounds):
    # Get data
    star_data = read_star_row_from_csv(star_name)
    
    # Initial guess: [Mass, Log10(Age), [Fe/H]]
    # Define bounds (Mass, Age, FeH)
    
    result = minimize(
        objective_function, 
        initial_guess, 
        args=(star_data,), 
        method='L-BFGS-B', 
        bounds=bounds
    )
    
    return result

In [ ]:
# For the Sun

In [10]:
# For HD 209458
HD_initial_guess = [1.1, 9.6, 0.0] 
HD_bounds = [(0.7, 1.5), (8.0, 10.0), (-0.5, 0.5)]

hd_result = estimate_age("HD 209458", HD_initial_guess, HD_bounds)
print(f"Results for HD 209458: {hd_result.x}")
print(f"Estimated Age (Gyr): {10**(hd_result.x[1]) / 1e9:.2f}")
print(f"Final Chi-Squared for HD 209458: {hd_result.fun:.2f}")

hd_result = estimate_age("HD 209458", HD_initial_guess, HD_bounds)
print(f"Success: {hd_result.success}")
print(f"Message: {hd_result.message}")
print(f"Final Chi-Squared: {hd_result.fun:.2e}") # Use scientific notation for big numbers

TypeError: '<' not supported between instances of 'NoneType' and 'float'

In [ ]:
# For HIP 8152
HIP_initial_guess = [0.937, 9.78, 0.007]
HIP_bounds = [(0.88, 1.00), (9.0, 10.1), (-0.15, 0.15)]

HIP_result = estimate_age("HIP 8152", HIP_initial_guess, HIP_bounds)
print(f"Results for HIP 8152: {HIP_result.x}")
print(f"Estimated Age (Gyr): {10**(HIP_result.x[1]) / 1e9:.2f}")
print(f"Final Chi-Squared for HIP 8152: {HIP_result.fun:.2f}")

OUT OF BOUNDS: M=0.94, Age=9.78, [Fe/H]=0.01
OUT OF BOUNDS: M=0.94, Age=9.78, [Fe/H]=0.01
OUT OF BOUNDS: M=0.94, Age=9.78, [Fe/H]=0.01
OUT OF BOUNDS: M=0.94, Age=9.78, [Fe/H]=0.01
Results for HIP 8152: [9.37e-01 9.78e+00 7.00e-03]
Estimated Age (Gyr): 6.03
Final Chi-Squared for HIP 8152: 10000000000.00


In [ ]:
# Test minimize to give back how it is choosing.

